# AquaCrisis LLM Classification Experiments

This notebook is for inspecting and smoke-testing the Mahti LLM experiment runner. For the full run, use the Slurm scripts.

The runner supports 12 experiments: Qwen 3.5, Gemma 4, and GPT-4.1 mini across zero-shot/5-shot and original/translated text. OpenAI experiments are batched and write JSONL, JSON, and CSV outputs.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('post_table_translation_preprocessed(2).csv')
OUTPUT_DIR = Path('llm_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Data path:', DATA_PATH)
print('Output directory:', OUTPUT_DIR)


In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.columns.tolist())
df[['post_id', 'native_text', 'translated_text']].head(3)


In [ ]:
import llm_classification_runner as runner

for i, exp in enumerate(runner.EXPERIMENTS):
    print(i, exp)


## Smoke Test One Experiment

For Ollama experiments, start Ollama first in a terminal with `ollama serve`. For OpenAI experiments, set `OPENAI_API_KEY` first.


In [ ]:
# Example smoke test command from inside a notebook.
# This runs only 5 posts and writes resumable outputs under llm_outputs_test/.

!python3 llm_classification_runner.py \
  --data "post_table_translation_preprocessed(2).csv" \
  --output-dir llm_outputs_test \
  --experiment gpt41mini_zero_translated \
  --limit 5 \
  --batch-size 5 \
  --resume


## Run One Full Experiment Manually

Use this only if you do not want to use Slurm arrays. For full Mahti runs, prefer the `.sh` scripts.


In [ ]:
EXPERIMENT_ID = 'gpt41mini_five_translated'

!python3 llm_classification_runner.py \
  --data "post_table_translation_preprocessed(2).csv" \
  --output-dir llm_outputs \
  --experiment {EXPERIMENT_ID} \
  --batch-size 20 \
  --resume


## Inspect Completed Outputs


In [ ]:
from pathlib import Path

for csv_path in sorted(Path('llm_outputs').glob('*/predictions.csv')):
    pred = pd.read_csv(csv_path)
    print(csv_path, pred.shape)
    print(pred['parse_ok'].value_counts(dropna=False).to_string())
    print()


## Combine All Prediction CSVs

This does not evaluate. It only creates one wide file with all available model predictions.


In [ ]:
base = pd.read_csv(DATA_PATH)[['post_id', 'native_text', 'translated_text']].drop_duplicates('post_id')
combined = base.copy()

for csv_path in sorted(Path('llm_outputs').glob('*/predictions.csv')):
    exp_id = csv_path.parent.name
    pred = pd.read_csv(csv_path)
    cols = ['post_id', 'task_a_label_id', 'task_a_label_name', 'task_b_label_id', 'task_b_label_name', 'confidence', 'parse_ok']
    pred = pred[[c for c in cols if c in pred.columns]].copy()
    pred = pred.rename(columns={c: f'{exp_id}_{c}' for c in pred.columns if c != 'post_id'})
    combined = combined.merge(pred, on='post_id', how='left')

combined_path = Path('llm_outputs/all_llm_predictions_combined.csv')
combined.to_csv(combined_path, index=False)
print('Saved:', combined_path)
print(combined.shape)
combined.head()
